# Battlesnake: Avoiding Immediate Danger

Let's take a look at how we need to respond to the [`move` HTTP POST request](https://docs.battlesnake.com/api/webhooks#response-properties-2).

```json
{
  "move": "up",
  "shout": "Moving up!" // this bit is optional
}
```

**We need to respond with a direction.**

Let's take a look at [an example `move` request payload](https://docs.battlesnake.com/api/example-move).

**We get all of our hazards in terms of coordinates.**

To determine if a move (direction) intersects with a hazard (coordinate), we need to be able to convert between the two.

## Objective 1: get_up_coord

Write a function that will take a coordinate and give us back the coordinate of the "up" direction. For example:

```python
up_coord = get_up_coord(head_coord={"x": 1, "y": 1})
```

WAIT! Don't start writing the function yet! Let's follow the precepts of Test Driven Development (TDD). We're not going to subscribe completely to TDD principals, but we're going to us an important one here:

1. Don't write any code until you have a failing test case

This might seem counterintuitive, but it forces us to think about testability from the inception of any change. If we wrote a functioning solution that is difficult - or impossible - to test, we might spend the next few days writing a test case fixture that requires more lines of code than the feature itself! Writing the test case first forces us to write the feature in a testable way from the first commit.

Let's write our first test case!


## Test Cases

Test cases in Python have come a long way. A lot of the Stackoverflow answers and documentation you'll find on the web will refer to a library called `unittests`. `unittests` was the de-facto testing library for Python for a long time. These days, all the cool kids are using the `pytest` library. Pytest is a large, mature library that deserves a much longer course with a much more qualified teacher. We'll cover basic concepts in this course. As you apply TDD to other code bases, you'll need to dive deeper into the library for more of its rich capabilities.

The starter repo already had some of the structure for writing tests. You'll find a top-level [tests](./tests) directory with a [test_snake](./tests/test_snake.py) file containing a __very__ simple test case: `test_info`. Let's run your test suite. First, **is your virtual environment activated?**

```bash
pytest .
```

You should see something like this:

```
=========================================================================================== test session starts ===========================================================================================
platform darwin -- Python 3.11.9, pytest-8.3.5, pluggy-1.5.0
rootdir: /Users/zanebclark/GitHubProjects/battlesnake-starter
configfile: pyproject.toml
plugins: anyio-4.9.0
collected 1 item

tests/test_snake.py .                                                                                                                                                                               [100%]

============================================================================================ 1 passed in 0.00s ============================================================================================
```

When you run `pytest .`, the library initially "collects" test cases from the directory you targeted. We targeted `.`, the directory that the terminal is currently at. Within that directory, the library searches for files that begin with `test_`. Within those files, it then searches for functions that begin with `test_` or classes that begin with `Test`.

After collecting your test cases, it will execute them and display the results.

We'll write all of our tests in the [tests](./tests) directory.
- First, it's where a rational person would look for tests. #berational
- Second, it will force the test cases to use the code in the [src](./src) directory as a library. That's not important to our program, but it will save you some heartache in the future if you write packages or libraries that you expect to be able to import and use.

So that Pytest can collect our tests, every file containing a test should start with `test_`. For now, let's add to our existing [test_snake](./tests/test_snake.py) file.

In [ ]:
def test_get_up_coord():  # The function name starts with "test_"
    # Given: The initial conditions
    head_coord = {"x": 1, "y": 1}

    # When: The action being tested. In this case, executing a function
    up_coord = get_up_coord(head_coord=head_coord)

    # Then: The expected outcome. In this case, we expect the return value of that function to be an explicit value.
    assert up_coord == {"x": 1, "y": 2}


Let's run the test case. Congratulations! You have a failing test case. You may now write some code. First, we need to define the function:

In [ ]:
def get_up_coord(head_coord: dict[str, int]) -> dict[str, int]:
    pass


Then, let's import the function in our [test_snake](./tests/test_snake.py) file:

In [ ]:
from battlesnakes.snake import info, get_up_coord


Run the test again. Now, we're getting a different error. That's exciting! I often develop with a single objective in mind: get a different error.

This time, Pytest is telling us that the return value doesn't match the expected value. Since we have a failing test case, we can write more code.

In [ ]:
def get_up_coord(head_coord: dict[str, int]) -> dict[str, int]:
    return {"x": head_coord["x"], "y": head_coord["y"] + 1}


Now our test case passes! We might want to test multiple cases to ensure it behaves as we would expect.

In [ ]:
def test_get_up_coord_2():  # The function name starts with "test_"
    # Given: The initial conditions
    head_coord = {"x": 9, "y": 9}

    # When: The action being tested. In this case, executing a function
    up_coord = get_up_coord(head_coord=head_coord)

    # Then: The expected outcome. In this case, we expect the return value of that function to be an explicit value.
    assert up_coord == {"x": 9, "y": 10}


That works too! A lot of that code is repetitive. Let's __parameterize__ the test case so we can reuse most of the code.

In [ ]:
import pytest


@pytest.mark.parametrize(
    "test_input,expected",
    [
        ({"x": 1, "y": 1}, {"x": 1, "y": 2}),
        ({"x": 9, "y": 9}, {"x": 9, "y": 10}),
    ],
)
def test_get_up_coord_parameterized(test_input, expected):
    up_coord = get_up_coord(head_coord=test_input)
    assert up_coord == expected


This does the same thing as our previous two test cases combined! The test case names don't tell us much about what we're testing. Let's fix that:

In [ ]:
def name_coordinate_test_cases(value):
    return f"({value['x']}, {value['y']})"


@pytest.mark.parametrize(
    "test_input,expected",
    [
        ({"x": 1, "y": 1}, {"x": 1, "y": 2}),
        ({"x": 9, "y": 9}, {"x": 9, "y": 10}),
    ],
    ids=name_coordinate_test_cases,
)
def test_get_up_coord_parameterized(test_input, expected):
    up_coord = get_up_coord(head_coord=test_input)
    assert up_coord == expected


Test cases are a great way to ensure that the code works the way you would expect it to given ideal circumstances. This is often referred to as testing the "happy path". It is equally important to test how your code responds to the "unhappy path". For example, we're assuming that the `head_coord` is a dictionary with "x" and "y" keys. If that's not the case, do we handle it gracefully? Let's find out.

In [1]:
def test_get_up_coord_not_a_valid_coordinate():
    head_coord = {"y": 9}

    up_coord = get_up_coord(head_coord=head_coord)

    assert up_coord == {"x": 9, "y": 10}


Because this is a test case, we get a nice printout of the input variable value. If this happened in the wild, we would just see `KeyError: 'x'` and have no way to understand which coordinate was a problem. Let's fix that:

In [ ]:
def get_up_coord(head_coord: dict[str, int]) -> dict[str, int]:
    if "x" not in head_coord.keys() or "y" not in head_coord.keys():
        raise ValueError(f"head_coord must have both 'x' and 'y' keys: {head_coord}")

    return {"x": head_coord["x"], "y": head_coord["y"] + 1}


If we rerun our test case, we'll get a much more verbose error: `ValueError: head_coord must have both 'x' and 'y' keys: {'y': 9}`. How do we communicate to Pytest that we expect this exception to be raised?

In [ ]:
def test_get_up_coord_not_a_valid_coordinate():
    head_coord = {"y": 9}

    with pytest.raises(
            ValueError, match="head_coord must have both 'x' and 'y' keys: .*"
    ):
        get_up_coord(head_coord=head_coord)

Here, we're asserting that an Exception is raised of a specific type with a specific error message. This test case passes. Beware of test cases that pass immediately. Let's alter it a bit to make sure it fails when it should:

In [ ]:
def test_get_up_coord_not_a_valid_coordinate():
    head_coord = {"y": 9}

    with pytest.raises(ValueError, match="the wrong message"):
        get_up_coord(head_coord=head_coord)

That fails, complaining that the message of the exception doesn't match the regular expression that we supplied. Let's return the correct message:

In [ ]:
def test_get_up_coord_not_a_valid_coordinate():
    head_coord = {"y": 9}

    with pytest.raises(
            ValueError, match="head_coord must have both 'x' and 'y' keys: .*"
    ):
        get_up_coord(head_coord=head_coord)


## Objective 2: get_direction_coord

Our `get_up_coord` function works well. Now, we might move on to writing a few more variants: `get_down_coord`, `get_left_coord`, and `get_right_coord`. Let's try `get_down_coord`:

In [ ]:
def get_down_coord(head_coord: dict[str, int]) -> dict[str, int]:
    if "x" not in head_coord.keys() or "y" not in head_coord.keys():
        raise ValueError(f"head_coord must have both 'x' and 'y' keys: {head_coord}")

    return {"x": head_coord["x"], "y": head_coord["y"] - 1}

There are only two differences here:
1. The name of the function
2. I've replaced `head_coord["y"] - 1` with `head_coord["y"] + 1`

The test cases would be similar as well. Could we write a `get_direction_coord` that would accept a `direction` and a `head_coord` and give us back the correct coordinate?

In [ ]:
def get_direction_coord(direction: str, head_coord: dict[str, int]) -> dict[str, int]:
    if "x" not in head_coord.keys() or "y" not in head_coord.keys():
        raise ValueError(f"head_coord must have both 'x' and 'y' keys: {head_coord}")

    match direction:
        case "up":
            return {"x": head_coord["x"], "y": head_coord["y"] + 1}
        case "down":
            return {"x": head_coord["x"], "y": head_coord["y"] - 1}
        case "left":
            return {"x": head_coord["x"] - 1, "y": head_coord["y"]}
        case "right":
            return {"x": head_coord["x"] + 1, "y": head_coord["y"]}


With a few tweaks, we can retrofit our existing parameterized test case for the new function:

In [ ]:
def name_coordinate_test_cases(value):
    if isinstance(value, dict):
        return f"({value['x']}, {value['y']})"
    return value


@pytest.mark.parametrize(
    "direction, head_coord,expected",
    [
        ("up", {"x": 1, "y": 1}, {"x": 1, "y": 2}),
        ("up", {"x": 9, "y": 9}, {"x": 9, "y": 10}),
    ],
    ids=name_coordinate_test_cases,
)
def test_get_direction_coord_parameterized(direction, head_coord, expected):
    up_coord = get_direction_coord(direction=direction, head_coord=head_coord)
    assert up_coord == expected

That works for the "up" cases. We can now easily add additional cases and test them in bulk:

In [ ]:
@pytest.mark.parametrize(
    "direction, head_coord,expected",
    [
        ("up", {"x": 1, "y": 1}, {"x": 1, "y": 2}),
        ("down", {"x": 1, "y": 1}, {"x": 1, "y": 0}),
        ("left", {"x": 1, "y": 1}, {"x": 0, "y": 1}),
        ("right", {"x": 1, "y": 1}, {"x": 2, "y": 1}),
        ("up", {"x": 9, "y": 9}, {"x": 9, "y": 10}),
        ("down", {"x": 9, "y": 9}, {"x": 9, "y": 8}),
        ("left", {"x": 9, "y": 9}, {"x": 8, "y": 9}),
        ("right", {"x": 9, "y": 9}, {"x": 10, "y": 9}),
    ],
    ids=name_coordinate_test_cases,
)
def test_get_direction_coord_parameterized(direction, head_coord, expected):
    up_coord = get_direction_coord(direction=direction, head_coord=head_coord)
    assert up_coord == expected


The "happy path" looks good. Let's test an "unhappy path". What if we supply an invalid direction? Note that we're updating our previous "unhappy path" test case to refer to our new `get_direction_coord` function:

In [ ]:
def test_get_direction_coord_not_a_valid_coordinate():
    with pytest.raises(
            ValueError, match="head_coord must have both 'x' and 'y' keys: .*"
    ):
        get_direction_coord(direction="up", head_coord={"y": 9})


def test_get_direction_coord_not_a_valid_direction():
    with pytest.raises(
            ValueError, match="head_coord must have both 'x' and 'y' keys: .*"
    ):
        get_direction_coord(direction="not a direction", head_coord={"x": 9, "y": 9})


The `test_get_direction_coord_not_a_valid_direction` should likely complain about the invalid direction, but it doesn't yet. Let's fix that:

In [ ]:
def get_direction_coord(direction: str, head_coord: dict[str, int]) -> dict[str, int]:
    if direction not in ["up", "down", "left", "right"]:
        raise ValueError(
            f"direction must be 'up', 'down', 'left', 'right': {direction}"
        )

    if "x" not in head_coord.keys() or "y" not in head_coord.keys():
        raise ValueError(f"head_coord must have both 'x' and 'y' keys: {head_coord}")

    match direction:
        case "up":
            return {"x": head_coord["x"], "y": head_coord["y"] + 1}
        case "down":
            return {"x": head_coord["x"], "y": head_coord["y"] - 1}
        case "left":
            return {"x": head_coord["x"] - 1, "y": head_coord["y"]}
        case "right":
            return {"x": head_coord["x"] + 1, "y": head_coord["y"]}

Let's update the message on the test case:

In [ ]:
def test_get_direction_coord_not_a_valid_direction():
    with pytest.raises(
            ValueError, match="direction must be 'up', 'down', 'left', 'right':.*"
    ):
        get_direction_coord(direction="not a direction", head_coord={"x": 9, "y": 9})


This works, but note that we have to list the directions twice: once for the `match` statement and once for the exception logic. Let's fix that by using the "default" case to raise an exception. Your IDE is also satisfied with this solution.

In [ ]:
def get_direction_coord(direction: str, head_coord: dict[str, int]) -> dict[str, int]:
    if "x" not in head_coord.keys() or "y" not in head_coord.keys():
        raise ValueError(f"head_coord must have both 'x' and 'y' keys: {head_coord}")

    match direction:
        case "up":
            return {"x": head_coord["x"], "y": head_coord["y"] + 1}
        case "down":
            return {"x": head_coord["x"], "y": head_coord["y"] - 1}
        case "left":
            return {"x": head_coord["x"] - 1, "y": head_coord["y"]}
        case "right":
            return {"x": head_coord["x"] + 1, "y": head_coord["y"]}
        case _:
            raise ValueError(
                f"direction must be 'up', 'down', 'left', 'right': {direction}"
            )


## Objective 3: get_possible_move_coords

Let's write a function that will give us back all potential move coordinates from our `head_coord`. First, let's write a test case:

In [ ]:
def test_get_possible_move_coords():
    head_coord = {"x": 1, "y": 1}

    possible_move_coords = get_possible_move_coords(head_coord=head_coord)


Now we have a failing test case. Let's go create that function. First, let's think about how to return the results. We're going to need to reach for a container type: `set`, `list`, `dict`, `tuple`. My earlier advice was to choose a type and change the type as you run into limitations. I'm going to pick the `set` type. Feel free to write it returning whichever type you prefer.


In [ ]:
def get_possible_move_coords(head_coord: dict[str, int]) -> set[dict[str, int]]:
    return {
        get_direction_coord(direction="up", head_coord=head_coord),
        get_direction_coord(direction="down", head_coord=head_coord),
        get_direction_coord(direction="left", head_coord=head_coord),
        get_direction_coord(direction="right", head_coord=head_coord),
    }

In [2]:
def test_get_possible_move_coords():
    head_coord = {"x": 1, "y": 1}

    possible_move_coords = get_possible_move_coords(head_coord=head_coord)
    assert possible_move_coords == {
        {"x": 1, "y": 2},
        {"x": 1, "y": 0},
        {"x": 0, "y": 1},
        {"x": 2, "y": 1},
    }


When you run it, you're going to get an exception: `TypeError: unhashable type: 'dict'`. We can talk about why this won't work, but I'd rather you **use this as an opportunity to sharpen your debugging skills**. This isn't the first time that you're going to be using a class that doesn't behave how you expect it to. Here are a few of my favorite sources to go to for specific errors:
1. Write an email to yourself explaining the problem. Write it as though you're going to send it to a trusted colleague. This colleague doesn't know the code or your objective. Explain the setup, the specific error, and what has changed since it last worked successfully. In the business, we call this [rubber duck debugging](https://en.wikipedia.org/wiki/Rubber_duck_debugging). By explaining the problem, you will review details that might help you resolve your own error. If you finish your email and are still stuck, you have a well-written "please help" email to send to a colleague.
2. The documentation behind the library you're using is a good first stop. In this case, we're not sure which concept is causing the error. I don't want you to crawl through all of Python's standard library documentation looking for this error.
3. Stackoverflow.com is a classic resource for programming questions and answers. The quality of answers on stackoverflow ranges from incredible to terrible. Be wary of what you read there. Despite this disclaimer, it is a very valuable debugging resource.
4. Be careful with generative AI tools like ChatGPT. First, ensure you are using a tool that your organization has approved. When you share a code snippet for feedback, you are technically sharing intellectual property with a 3rd party. Make sure it's an approved transaction. Second, ChatGPT's hallucinations are incredibly persuasive. When a Stackoverflow answer is invalid, there's a general lack of documentation or code snippets. Comments on the answer may warn you with statements like "this didn't work for me." ChatGPT will create cited code snippets for completely imaginary concepts. What's more, there's no community feedback on the freshly-generated answer.

```



   ,---,                   ,---,.    ,---,.           .---.
  '  .' \                ,'  .' |  ,'  .' |          /. ./|
 /  ;    '.            ,---.'   |,---.'   |      .--'.  ' ;
:  :       \           |   |   .'|   |   .'     /__./ \ : |
:  |   /\   \          :   :  :  :   :  |-, .--'.  '   \' .
|  :  ' ;.   :         :   |  |-,:   |  ;/|/___/ \ |    ' '
|  |  ;/  \   \        |   :  ;/||   :   .';   \  \;      :
'  :  | \  \ ,'        |   |   .'|   |  |-, \   ;  `      |
|  |  '  '--'          '   :  '  '   :  ;/|  .   \    .\  ;
|  :  :                |   |  |  |   |    \   \   \   ' \ |
|  | ,'                |   :  \  |   :   .'    :   '  |--"
`--''                  |   | ,'  |   | ,'       \   \ ;
                       `----'    `----'          '---"                       ,----,
          ____      ,----..             ____                     ,--.      ,/   .`|
        ,'  , `.   /   /   \          ,'  , `.    ,---,.       ,--.'|    ,`   .'  : .--.--.
     ,-+-,.' _ |  /   .     :      ,-+-,.' _ |  ,'  .' |   ,--,:  : |  ;    ;     //  /    '.
  ,-+-. ;   , || .   /   ;.  \  ,-+-. ;   , ||,---.'   |,`--.'`|  ' :.'___,/    ,'|  :  /`. /
 ,--.'|'   |  ;|.   ;   /  ` ; ,--.'|'   |  ;||   |   .'|   :  :  | ||    :     | ;  |  |--`
|   |  ,', |  ':;   |  ; \ ; ||   |  ,', |  '::   :  |-,:   |   \ | :;    |.';  ; |  :  ;_
|   | /  | |  |||   :  | ; | '|   | /  | |  ||:   |  ;/||   : '  '; |`----'  |  |  \  \    `.
'   | :  | :  |,.   |  ' ' ' :'   | :  | :  |,|   :   .''   ' ;.    ;    '   :  ;   `----.   \
;   . |  ; |--' '   ;  \; /  |;   . |  ; |--' |   |  |-,|   | | \   |    |   |  '   __ \  \  |
|   : |  | ,     \   \  ',  / |   : |  | ,    '   :  ;/|'   : |  ; .'    '   :  |  /  /`--'  /
|   : '  |/       ;   :    /  |   : '  |/     |   |    \|   | '`--'      ;   |.'  '--'.     /
;   | |`-'         \   \ .'   ;   | |`-'      |   :   .''   : |          '---'      `--'---'
|   ;/              `---`     |   ;/          |   | ,'  ;   |.'
'---'                         '---'           `----'    '---'
   ,--,                         ,----,
,---.'|                       ,/   .`|
|   | :      ,---,          ,`   .'  :   ,---,.,-.----.
:   : |     '  .' \       ;    ;     / ,'  .' |\    /  \
|   ' :    /  ;    '.   .'___,/    ,',---.'   |;   :    \
;   ; '   :  :       \  |    :     | |   |   .'|   | .\ :
'   | |__ :  |   /\   \ ;    |.';  ; :   :  |-,.   : |: |
|   | :.'||  :  ' ;.   :`----'  |  | :   |  ;/||   |  \ :
'   :    ;|  |  ;/  \   \   '   :  ; |   :   .'|   : .  /
|   |  ./ '  :  | \  \ ,'   |   |  ' |   |  |-,;   | |  \
;   : ;   |  |  '  '--'     '   :  | '   :  ;/||   | ;\  \
|   ,/    |  :  :           ;   |.'  |   |    \:   ' | \.'
'---'     |  | ,'           '---'    |   :   .':   : :-'
          `--''                      |   | ,'  |   |.'
                                     `----'    `---'

```

Did you figure it out? Great!

If not, here's the short answer: We can't store "mutable" objects in sets. Dictionaries are "mutable" objects. We can refactor our return container to be a list or a container. I'm using a list below.

Here's [the long answer](https://youtu.be/Sd8BP922RUg?si=x4iHkTL6xlLAD6nd&t=331)


In [ ]:
# snake.py
def get_possible_move_coords(head_coord: dict[str, int]) -> list[dict[str, int]]:
    return [
        get_direction_coord(direction="up", head_coord=head_coord),
        get_direction_coord(direction="down", head_coord=head_coord),
        get_direction_coord(direction="left", head_coord=head_coord),
        get_direction_coord(direction="right", head_coord=head_coord),
    ]

# test_snake.py
def test_get_possible_move_coords():
    head_coord = {"x": 1, "y": 1}

    possible_move_coords = get_possible_move_coords(head_coord=head_coord)
    assert possible_move_coords == [
        {"x": 1, "y": 0},
        {"x": 1, "y": 2},
        {"x": 0, "y": 1},
        {"x": 2, "y": 1},
    ]


I'm now returning a list and expecting a list. I made a change that will cause the test case to fail: I switched the order of two of the variables in the assert statement. Lists retain their order. When we assert list equality, a difference in the order will cause the assertion to fail. We don't __really__ care about the order, so let's ignore the order in the test case:

In [ ]:
def test_get_possible_move_coords():
    head_coord = {"x": 1, "y": 1}
    expected_move_coords: list[dict[str, int]] = [
        {"x": 1, "y": 0},
        {"x": 1, "y": 2},
        {"x": 0, "y": 1},
        {"x": 2, "y": 1},
    ]

    possible_move_coords = get_possible_move_coords(head_coord=head_coord)
    assert len(possible_move_coords) == len(expected_move_coords)
    for expected_move_coord in expected_move_coords:
        assert expected_move_coord in possible_move_coords


## Objective 4: is_coord_on_board

All of our hazards are expressed as coordinates and all of our moves __were__ expressed as directions. We've converted our moves into coordinates (`get_possible_move_coords`). Now, we can compare the hazard coordinates to our move coordinates and eliminate the overlaps. We're going to start with moves that would take us off the edge of the map.

Technically, the out-of-bounds hazards aren't expressed as coordinates. Rather, we get the height (max y coordinate) and width (max x coordinate) of the board from the [board object](https://docs.battlesnake.com/api/objects/board) on the [`move` payload](https://docs.battlesnake.com/api/example-move). For example:

```json
{
  "height": 11,
  "width": 11,
  "food": [
    {"x": 5, "y": 5},
    {"x": 9, "y": 0},
    {"x": 2, "y": 6}
  ],
  "hazards": [
    {"x": 0, "y": 0},
    {"x": 0, "y": 1},
    {"x": 0, "y": 2}
  ],
  "snakes": [
    {"id": "snake-one", ... },
    {"id": "snake-two", ... },
    {"id": "snake-three", ... }
  ]
}
```

Let's write a function that will take the board height, board width, and a coordinate. If the coordinate is on the board, it will return `True`. If it isn't, it will return `False`.

I tricked you!

We'll start writing the test case first. :)

In [ ]:
def test_is_coord_on_board():
    coord = {"x": 1, "y": 1}
    board_height = 11
    board_width = 11

    result = is_coord_on_board(
        coord=coord,
        board_height=board_height,
        board_width=board_width,
    )

    assert result == True


Let's get this test case passing:

In [ ]:
def is_coord_on_board(
        coord: dict[str, int],
        board_height: int,
        board_width: int,
) -> bool:
    if coord["x"] < 0 or coord["x"] >= board_width:
        return False
    if coord["y"] < 0 or coord["y"] >= board_height:
        return False
    return True


That works! It's easy to read. Another way to write it:

In [ ]:
def is_coord_on_board(
        coord: dict[str, int],
        board_height: int,
        board_width: int,
) -> bool:
    if (
            coord["x"] < 0
            or coord["x"] >= board_width
            or coord["y"] < 0
            or coord["y"] >= board_height
    ):
        return False
    return True


Yet another way:

In [ ]:
def is_coord_on_board(
        coord: dict[str, int],
        board_height: int,
        board_width: int,
) -> bool:
    return not (
            coord["x"] < 0
            or coord["x"] >= board_width
            or coord["y"] < 0
            or coord["y"] >= board_height
    )


Now, let's bulk up the test case by parameterizing it:

In [ ]:
@pytest.mark.parametrize(
    "coord, board_height, board_width, expected",
    [({"x": 1, "y": 1}, 11, 11, True)],
    ids=name_coordinate_test_cases,
)
def test_is_coord_on_board(
        coord: dict[str, int],
        board_height: int,
        board_width: int,
        expected: bool,
):
    result = is_coord_on_board(
        coord=coord,
        board_height=board_height,
        board_width=board_width,
    )

    assert result == expected

We might flip the expected `True` to `False` and run it just to make sure it fails. Good, that works. Now, let's add a few more cases to our parameterized test case:

In [ ]:
@pytest.mark.parametrize(
    "coord, board_height, board_width, expected",
    [
        ({"x": 0, "y": 0}, 11, 11, True),
        ({"x": 10, "y": 10}, 11, 11, True),
        ({"x": 0, "y": 10}, 11, 11, True),
        ({"x": 10, "y": 0}, 11, 11, True),
        ({"x": -1, "y": -1}, 11, 11, False),
        ({"x": 11, "y": 11}, 11, 11, False),
        ({"x": 0, "y": -1}, 11, 11, False),
        ({"x": 11, "y": 0}, 11, 11, False),
    ],
    ids=name_coordinate_test_cases,
)
def test_is_coord_on_board(
        coord: dict[str, int],
        board_height: int,
        board_width: int,
        expected: bool,
):
    result = is_coord_on_board(
        coord=coord,
        board_height=board_height,
        board_width=board_width,
    )

    assert result == expected


## Objective 5: is_coord_on_snake

Now, let's develop the ability to check if a move intersects with another snake. The [board object](https://docs.battlesnake.com/api/objects/board) on the [`move` payload](https://docs.battlesnake.com/api/example-move) has information about every snake on the board. Let's write a function that takes the move payload and a coordinate as parameters. If the coordinate is on a snake's current body, it will return `True`. If it isn't, it will return `False`.

I tricked you again!

We'll start writing the test case first. This test is a little more complicated than our previous tests. Out other functions have required simple inputs that we can define on a single line. This test will require a move payload. We might initially write something like this:


In [ ]:
def test_is_coord_on_snake():
    coord = {"x": 0, "y": 0}
    move_payload = {...}  # The ~50-line example payload here: https://docs.battlesnake.com/api/example-move

    result = is_coord_on_snake(coord=coord, move_payload=move_payload)
    assert result == True

Using the example [`move` payload](https://docs.battlesnake.com/api/example-move) in this test case makes it hard to read. Most of the lines of code aren't being used in the test case. Adding additional test cases will cause the size and complexity of the file to explode. There's a better way:

### Test Case Fixtures

It is common for test cases to require some setup or configuration. Often, this logic applies to multiple test cases. For example, testing a database application might  require a database connection for multiple test cases. The connection setup isn't being tested, so its presence in test cases is distracting. Pytest would have you pull that connection establishment logic out of the test cases and define a [test case fixture](https://docs.pytest.org/en/6.2.x/fixture.html). That fixture would be defined in a particular file and available to all test cases that include the fixture name as a parameter. Fixtures are very powerful and very configurable.

We're not going to define an __actual__ fixture for our `move` request object. Fixtures offer a lot of features with a jump in complexity. We don't need the complexity and we want to prioritize simplicity. Instead, we're going to define a function that returns a full request object. It will be close enough to a fixture to demonstrate the concept.



In [ ]:
def get_move_payload() -> dict:
    return {...}  # The ~50-line example payload here: https://docs.battlesnake.com/api/example-move

def test_is_coord_on_snake():
    coord = {"x": 0, "y": 0}
    move_payload = get_move_payload()

    result = is_coord_on_snake(coord=coord, move_payload=move_payload)
    assert result == True

Now, I can call the `get_move_payload` in every test case that requires a move payload. Let's define our function and get a different failure:

In [ ]:
def is_coord_on_snake(
        coord: dict[str, int],
        move_payload: dict,
):
    pass


We could reference the documentation to understand how to get the snake bodies from the payload, but not every library has stellar documentation. Instead, we're going to learn how to use breakpoints to debug your code.

### Breakpoints

Let's watch the video on Microsoft's official documentation on the subject: [Debug code with Visual Studio Code](https://code.visualstudio.com/docs/debugtest/debugging#:~:text=An%20inline%20breakpoint%20can%20be,in%20the%20editor's%20left%20margin.).

First, set a breakpoint in the function.
Second, run the test case in debug mode.

![01_pytest_debug](docs/assets/avoid_danger/01_pytest_debug.png)

The line where we placed our breakpoint should turn yellow and we should be able to inspect the variables that are in the stack at that point in execution. Let's find the "board" > "snakes" list. Let's use this to figure out exactly what we're going to do in the function. Create a "watch" on the left-hand side and paste the following: `move_payload["board"]["snakes"]`. Press enter. You're looking at the value at that point in the object! You can use this to rapidly experiment with different approaches. Note: If you alter the variable using this interface, it will be altered for the rest of the program's execution. I would recommend that you use it for "read only" operations to ensure you don't introduce any strange behavior.

We're going to want to iterate over each snake, so let's update the function definition and set a new breakpoint:

In [ ]:
def is_coord_on_snake(
        coord: dict[str, int],
        move_payload: dict,
):
    for snake in move_payload["board"]["snakes"]:
        pass


Now, we can get at the body using `snake["body"]`. We can even validate our logic for checking if the coordinate is in the snake body: `coord in snake["body"]`. This returns `True`, which is what we expected.

In [ ]:
def is_coord_on_snake(
        coord: dict[str, int],
        move_payload: dict,
):
    for snake in move_payload["board"]["snakes"]:
        if coord in snake["body"]:
            return True

Our test case passes! Let's parameterize the test case and add a few more cases.

In [ ]:
@pytest.mark.parametrize(
    "coord, expected",
    [
        ({"x": 0, "y": 0}, True),
        ({"x": 5, "y": 4}, True),
        ({"x": 1, "y": 1}, False),
    ],
    ids=name_coordinate_test_cases,
)
def test_is_coord_on_snake(coord: dict[str, int], expected: bool):
    move_payload = get_move_payload()
    result = is_coord_on_snake(coord=coord, move_payload=move_payload)
    assert result == expected


Now we get another failure. We forgot to return `False` if the coordinate isn't on a snake body:

In [ ]:
def is_coord_on_snake(
        coord: dict[str, int],
        move_payload: dict,
):
    for snake in move_payload["board"]["snakes"]:
        if coord in snake["body"]:
            return True
    return False

## Objective 6: get_safe_moves

We're almost there! Let's combine our three functions in a fourth function.
1. We'll get our head coordinate
2. We'll get our possible move coordinates with our `get_possible_move_coords` function.
3. We'll iterate over our potential moves and exclude them using the `is_coord_on_board` and `is_coord_on_snake` functions.
4. We'll return moves we consider safe

Let's start with a test case:


In [ ]:
def test_get_safe_moves():
    move_payload = get_move_payload()
    safe_moves = get_safe_moves(move_payload=move_payload)


Now, we'll write some code:

In [ ]:
def get_safe_moves(move_payload: dict):
    head_coord = move_payload["you"]["head"]
    board_height = move_payload["board"]["height"]
    board_width = move_payload["board"]["width"]

    move_coords = get_possible_move_coords(head_coord=head_coord)

    safe_moves = []
    for coord in move_coords:
        if not is_coord_on_board(
                coord=coord, board_height=board_height, board_width=board_width
        ):
            continue
        if is_coord_on_snake(coord=coord, move_payload=move_payload):
            continue
        safe_moves.append(coord)

    return safe_moves


Now, let's update our test case:

In [ ]:
def test_get_safe_moves():
    move_payload = get_move_payload()
    safe_moves = get_safe_moves(move_payload=move_payload)
    assert safe_moves == [{"x": 0, "y": 1}]

This test case passes! Now, let's increase the coverage by parameterizing the test. Doing so with previous tests was easier because we were able to pass in an arbitrary coordinate. In this case, the only interface we have is the move payload. We're going to have to parameterize the move payload. Let's add a parameter to the `get_move_payload` function that allows us to supply our snake's body.

In [ ]:
def get_move_payload(my_snake_body: list[dict[str, int]] | None = None) -> dict:
    ...

The `my_snake_body` parameter has a default value of `None`. When defining a function, you can supply a value that is used if one is not supplied. In this case, we want to avoid altering our previous test cases to pass in a snake body. By setting the default as `None`, we can use conditional logic to supply a default value for `my_snake_body` when one isn't supplied. Our existing test cases will continue to function properly.

In [ ]:
def get_move_payload(my_snake_body: list[dict[str, int]] | None = None) -> dict:
    if my_snake_body is None:
        my_snake_body = [{"x": 0, "y": 0}, {"x": 1, "y": 0}, {"x": 2, "y": 0}]

    return {
        "game": {
            "id": "totally-unique-game-id",
            "ruleset": {
                "name": "standard",
                "version": "v1.1.15",
                "settings": {
                    "foodSpawnChance": 15,
                    "minimumFood": 1,
                    "hazardDamagePerTurn": 14,
                },
            },
            "map": "standard",
            "source": "league",
            "timeout": 500,
        },
        "turn": 14,
        "board": {
            "height": 11,
            "width": 11,
            "food": [{"x": 5, "y": 5}, {"x": 9, "y": 0}, {"x": 2, "y": 6}],
            "hazards": [{"x": 3, "y": 2}],
            "snakes": [
                {
                    "id": "snake-508e96ac-94ad-11ea-bb37",
                    "name": "My Snake",
                    "health": 54,
                    "body": my_snake_body,
                    "latency": "111",
                    "head": {"x": 0, "y": 0},
                    "length": 3,
                    "shout": "why are we shouting??",
                    "customizations": {
                        "color": "#FF0000",
                        "head": "pixel",
                        "tail": "pixel",
                    },
                },
                {
                    "id": "snake-b67f4906-94ae-11ea-bb37",
                    "name": "Another Snake",
                    "health": 16,
                    "body": [
                        {"x": 5, "y": 4},
                        {"x": 5, "y": 3},
                        {"x": 6, "y": 3},
                        {"x": 6, "y": 2},
                    ],
                    "latency": "222",
                    "head": {"x": 5, "y": 4},
                    "length": 4,
                    "shout": "I'm not really sure...",
                    "customizations": {
                        "color": "#26CF04",
                        "head": "silly",
                        "tail": "curled",
                    },
                },
            ],
        },
        "you": {
            "id": "snake-508e96ac-94ad-11ea-bb37",
            "name": "My Snake",
            "health": 54,
            "body": my_snake_body,
            "latency": "111",
            "head": {"x": 0, "y": 0},
            "length": 3,
            "shout": "why are we shouting??",
            "customizations": {"color": "#FF0000", "head": "pixel", "tail": "pixel"},
        },
    }

Let's run our test case suite again to verify that we didn't break any existing test cases.

Good. What if we accidentally define a snake body that's not possible. For example, a snake body with the following coordinates would be impossible:

```
[{"x": 0, "y": 0}, {"x": 1, "y": 1}]
```

Let's write a small function to validate coordinate adjacency so we can avoid fabricating invalid scenarios:

In [ ]:
def check_coordinate_adjacency(coords: list[dict[str, int]]) -> None:
    for index, current_coord in enumerate(coords):
        try:
            next_coord = coords[index + 1]
            x_delta = current_coord["x"] - next_coord["x"]
            y_delta = current_coord["y"] - next_coord["y"]
            if abs(x_delta + y_delta) != 1:
                raise Exception(
                    f"coordinates aren't adjacent: {current_coord}, {next_coord}"
                )
        except IndexError:
            continue

Now, let's use that function in our `get_move_payload` function:

In [ ]:
def get_move_payload(my_snake_body: list[dict[str, int]] | None = None) -> dict:
    if my_snake_body is None:
        my_snake_body = [{"x": 0, "y": 0}, {"x": 1, "y": 0}, {"x": 2, "y": 0}]

    check_coordinate_adjacency(coords=my_snake_body)

    return {
        "game": {
            "id": "totally-unique-game-id",
            "ruleset": {
                "name": "standard",
                "version": "v1.1.15",
                "settings": {
                    "foodSpawnChance": 15,
                    "minimumFood": 1,
                    "hazardDamagePerTurn": 14,
                },
            },
            "map": "standard",
            "source": "league",
            "timeout": 500,
        },
        "turn": 14,
        "board": {
            "height": 11,
            "width": 11,
            "food": [{"x": 5, "y": 5}, {"x": 9, "y": 0}, {"x": 2, "y": 6}],
            "hazards": [{"x": 3, "y": 2}],
            "snakes": [
                {
                    "id": "snake-508e96ac-94ad-11ea-bb37",
                    "name": "My Snake",
                    "health": 54,
                    "body": my_snake_body,
                    "latency": "111",
                    "head": {"x": 0, "y": 0},
                    "length": 3,
                    "shout": "why are we shouting??",
                    "customizations": {
                        "color": "#FF0000",
                        "head": "pixel",
                        "tail": "pixel",
                    },
                },
                {
                    "id": "snake-b67f4906-94ae-11ea-bb37",
                    "name": "Another Snake",
                    "health": 16,
                    "body": [
                        {"x": 5, "y": 4},
                        {"x": 5, "y": 3},
                        {"x": 6, "y": 3},
                        {"x": 6, "y": 2},
                    ],
                    "latency": "222",
                    "head": {"x": 5, "y": 4},
                    "length": 4,
                    "shout": "I'm not really sure...",
                    "customizations": {
                        "color": "#26CF04",
                        "head": "silly",
                        "tail": "curled",
                    },
                },
            ],
        },
        "you": {
            "id": "snake-508e96ac-94ad-11ea-bb37",
            "name": "My Snake",
            "health": 54,
            "body": my_snake_body,
            "latency": "111",
            "head": {"x": 0, "y": 0},
            "length": 3,
            "shout": "why are we shouting??",
            "customizations": {"color": "#FF0000", "head": "pixel", "tail": "pixel"},
        },
    }


To validate our `check_coordinate_adjacency` function, let's write two test cases:

In [ ]:
def test_check_coordinate_adjacency_happy_path():
    coords = [{"x": 0, "y": 0}, {"x": 1, "y": 0}, {"x": 2, "y": 0}]
    check_coordinate_adjacency(coords=coords)
    assert True


def test_check_coordinate_adjacency_not_adjacent():
    coords = [{"x": 0, "y": 0}, {"x": 1, "y": 1}, {"x": 2, "y": 0}]
    with pytest.raises(Exception, match="coordinates aren't adjacent: .*"):
        check_coordinate_adjacency(coords=coords)
    assert True


Now, it's possible that we define two overlapping snake bodies. Let's add a check to the `get_move_payload` to ensure there's no overlap:

In [ ]:
def get_move_payload(my_snake_body: list[dict[str, int]] | None = None) -> dict:
    if my_snake_body is None:
        my_snake_body = [{"x": 0, "y": 0}, {"x": 1, "y": 0}, {"x": 2, "y": 0}]
    check_coordinate_adjacency(coords=my_snake_body)

    other_snake_body = [
        {"x": 5, "y": 4},
        {"x": 5, "y": 3},
        {"x": 6, "y": 3},
        {"x": 6, "y": 2},
    ]
    check_coordinate_adjacency(coords=other_snake_body)

    for my_coord in my_snake_body:
        if my_coord in other_snake_body:
            raise Exception(
                f"snake body coordinates overlap:\n{my_snake_body}\n{other_snake_body}"
            )

    return {
        "game": {
            "id": "totally-unique-game-id",
            "ruleset": {
                "name": "standard",
                "version": "v1.1.15",
                "settings": {
                    "foodSpawnChance": 15,
                    "minimumFood": 1,
                    "hazardDamagePerTurn": 14,
                },
            },
            "map": "standard",
            "source": "league",
            "timeout": 500,
        },
        "turn": 14,
        "board": {
            "height": 11,
            "width": 11,
            "food": [{"x": 5, "y": 5}, {"x": 9, "y": 0}, {"x": 2, "y": 6}],
            "hazards": [{"x": 3, "y": 2}],
            "snakes": [
                {
                    "id": "snake-508e96ac-94ad-11ea-bb37",
                    "name": "My Snake",
                    "health": 54,
                    "body": my_snake_body,
                    "latency": "111",
                    "head": {"x": 0, "y": 0},
                    "length": 3,
                    "shout": "why are we shouting??",
                    "customizations": {
                        "color": "#FF0000",
                        "head": "pixel",
                        "tail": "pixel",
                    },
                },
                {
                    "id": "snake-b67f4906-94ae-11ea-bb37",
                    "name": "Another Snake",
                    "health": 16,
                    "body": other_snake_body,
                    "latency": "222",
                    "head": {"x": 5, "y": 4},
                    "length": 4,
                    "shout": "I'm not really sure...",
                    "customizations": {
                        "color": "#26CF04",
                        "head": "silly",
                        "tail": "curled",
                    },
                },
            ],
        },
        "you": {
            "id": "snake-508e96ac-94ad-11ea-bb37",
            "name": "My Snake",
            "health": 54,
            "body": my_snake_body,
            "latency": "111",
            "head": {"x": 0, "y": 0},
            "length": 3,
            "shout": "why are we shouting??",
            "customizations": {"color": "#FF0000", "head": "pixel", "tail": "pixel"},
        },
    }


Let's write a function to validate this behavior:

In [ ]:
def test_get_move_payload_overlap():
    my_snake_body = [{"x": 5, "y": 4}, {"x": 5, "y": 5}, {"x": 5, "y": 6}]
    with pytest.raises(Exception, match="snake body coordinates overlap:.*"):
        get_move_payload(my_snake_body=my_snake_body)


Now, we can parameterize our `test_get_safe_moves` function:

In [ ]:
@pytest.mark.parametrize(
    "my_snake_body, expected",
    [
        ([{"x": 0, "y": 0}, {"x": 1, "y": 0}, {"x": 2, "y": 0}], [{"x": 0, "y": 1}]),
    ],
    ids=name_coordinate_test_cases,
)
def test_get_safe_moves(my_snake_body, expected):
    move_payload = get_move_payload(my_snake_body=my_snake_body)
    safe_moves = get_safe_moves(move_payload=move_payload)
    assert safe_moves == expected


We won't add any additional test cases to this list in this session, but I would encourage you to write at least two additional test cases with unique situations

## Objective 7: Return a Direction

We're so close! We've traded directions for coordinates. We've filtered out hazardous coordinates from our available coordinates. Now, we have a list of safe coordinates to move to. Let's return one! Hold on a minute, we need to return a direction. How do we get a direction from our safe move coordinates? We could compare the safe coordinates to our head coordinate and use that to back into a direction. It feels like we already did the opposite of that in `get_possible_move_coords`. Could we associate the direction with the coordinate so that it is there when we finish filtering out unsafe coordinates?

I did this on purpose to simulate a very common situation in programming. You're making progress towards an objective and you need to make a change to some of the code that you've already written. This is called "refactoring". Refactoring can be very painful. Luckily, we've been using a number of tools that will make the process easier:

1. git: Commit your current state. Create a feature branch for the refactor. If you decide to turn back at any point or want to compare the previous state to the refactored state, you can compare the refactor branch to the initial feature branch.
2. Test Cases: If you validate functionality manually, refactoring your code should trigger a manual retest. Since we've been writing test cases, we can immediately re-test the entire codebase with the click of a button.
3. IDEs: If you mess up the refactor in some way, your IDE will "complain" by underlining parts of the code it thinks are wrong. Use this to check your work.

Let's change the return type of the `get_possible_move_coords` function to a dictionary and use the direction as a key and the coordinate as a value. While we're at it, I'll introduce you to dictionary comprehension. Dictionary comprehension is a very succinct - and performant - way to create a dictionary. Start by iterating over something and creating a key-value pair within the loop:

```
for direction in ["up", "down", "left", "right"]:
    direction: get_direction_coord(direction=direction
```

This looks pretty familiar, right? We've iterated over lists before. We're familiar with a dictionary's key-value pairs. To turn this into a dictionary comprehension, you just need to change the order and wrap it in a set of curly braces:

```
{
    direction: get_direction_coord(direction=direction, head_coord=head_coord)
    for direction in ["up", "down", "left", "right"]
}
```

The order is a bit counterintuitive, as we're used to looping over things first and doing something inside of the loop that follows. In this case, we define what we want to happen in the loop __before__ you define the loop.

Dictionary comprehensions are good to use because they are fast. They can make code harder to read, so balance readability with performance.

In [ ]:
def get_possible_move_coords(head_coord: dict[str, int]) -> dict[str, dict[str, int]]:
    return {
        direction: get_direction_coord(direction=direction, head_coord=head_coord)
        for direction in ["up", "down", "left", "right"]
    }

We need to update some test cases to reflect this new reality:

In [ ]:
def test_get_possible_move_coords():
    head_coord = {"x": 1, "y": 1}
    expected_move_coords: dict[str, dict[str, int]] = {
        "down": {"x": 1, "y": 0},
        "up": {"x": 1, "y": 2},
        "left": {"x": 0, "y": 1},
        "right": {"x": 2, "y": 1},
    }

    possible_move_coords = get_possible_move_coords(head_coord=head_coord)
    assert len(possible_move_coords) == len(expected_move_coords)
    for move, expected_move_coord in expected_move_coords.items():
        assert possible_move_coords[move] == expected_move_coord

@pytest.mark.parametrize(
    "my_snake_body, expected",
    [
        ([{"x": 0, "y": 0}, {"x": 1, "y": 0}, {"x": 2, "y": 0}], ["up"]),
    ],
    ids=name_coordinate_test_cases,
)
def test_get_safe_moves(my_snake_body, expected):
    move_payload = get_move_payload(my_snake_body=my_snake_body)
    safe_moves = get_safe_moves(move_payload=move_payload)
    assert safe_moves == expected


## Objective 8: Think In Terms Of The Next Turn

Our `get_safe_moves` is good at finding safe moves on the current board, but our move will be considered in the context of the next turn's board. Each snake will move, causing the current tail location to disappear and the head to be "drawn" on one of the available move slots.

Right now, we're considering the tail of each snake to be dangerous. It's not. Can you alter the `is_coord_on_snake` function to ignore the last coordinate of each snake body?

We're also considering the area around the head of other snakes to be safe. It's not entirely safe. There's some chance that this area will be occupied by another snake head. If that snake is the same size as our snake or larger, that will cause our snake to be eliminated. Can you alter the `is_coord_on_snake` function to use `get_possible_move_coords` to consider potential move coordinates for snakes that are a threat to you?

